# HF2VAD — UCSDped2 — Train & Eval AUC (RAFT flows)

Pipeline đầy đủ theo repo [1402_ImageNet-Big-Application](https://github.com/ngthhaiha/1402_ImageNet-Big-Application), sử dụng **optical flow .npy đã có sẵn** (RAFT) — không extract lại.

```
STAGE 1 ▸ Setup & cấu hình đường dẫn
STAGE 2 ▸ Ánh xạ dữ liệu Kaggle → cấu trúc HF2VAD chuẩn
STAGE 3 ▸ Extract BBoxes (Faster R-CNN + fg motion)
STAGE 4 ▸ Build Chunked Samples .pkl (frames + flows + bboxes)
STAGE 5 ▸ Định nghĩa Models (ML-MemAE-SC + VUNet)
STAGE 6 ▸ Train Stage-1: ML-MemAE-SC (flow reconstruction)
STAGE 7 ▸ Train Stage-2: HFVAD full model (frame prediction)
STAGE 8 ▸ Evaluate AUC + Visualize anomaly score curves
```

**Kaggle inputs:**
- `hihnguynth/ucsd-anomaly-dataset` → UCSDped2 frames (.tif)
- `lmnguynpht/optical-flow-ped2-ucsd` → optical flows (.npy, RAFT)

**Ước tính thời gian** (GPU T4, `EPOCHS_MEMAE=2`, `EPOCHS_HFVAD=3`):
BBox ~5min | Chunked ~8min | Train-S1 ~8min | Train-S2 ~12min | Eval ~3min

## STAGE 1 — Setup

In [ ]:
!pip install tensorboardX scikit-learn joblib tqdm --quiet

In [ ]:
import os, sys, gc, re, glob, math, shutil, pickle
from pathlib import Path
from collections import OrderedDict

import cv2
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn import ModuleDict, ModuleList, Conv2d
from torch.nn.utils import weight_norm
import torchvision.transforms as tvT
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as TF
from tqdm import tqdm
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc
import scipy.signal as ss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================
#  ⚙️  CẤU HÌNH — chỉnh tại đây nếu path khác
# ==============================================================

# ---- Kaggle input paths ----
FRAMES_BASE = Path("/kaggle/input/datasets/hihnguynth/ucsd-anomaly-dataset"
                   "/UCSD_Anomaly_Dataset.v1p2/UCSDped2")
FLOWS_BASE  = Path("/kaggle/input/datasets/lmnguynpht/optical-flow-ped2-ucsd"
                   "/optical_flow_results/UCSDped2")

# ---- Working directory ----
WORK        = Path("/kaggle/working")
DATA_DIR    = WORK / "data" / "ped2"   # cấu trúc chuẩn HF2VAD
CKPT_DIR    = WORK / "ckpt"
EVAL_DIR    = WORK / "eval"

# ---- Hyperparameters ----
EPOCHS_MEMAE = 2     # Stage-1 MemAE  (paper: 80)
EPOCHS_HFVAD = 3     # Stage-2 HFVAD  (paper: 50)
BATCH_SIZE   = 128
LR           = 1e-4
SAVE_EVERY   = 1     # eval mỗi N epoch
NUM_WORKERS  = 2

# ---- Model config (giống cfgs/cfg.yaml gốc) ----
MODEL_PARAS = dict(
    final_act=False, nf_max=128, nf_start=64, spatial_size=32,
    dropout_prob=0.1, img_channels=3, motion_channels=2,
    clip_hist=4, clip_pred=1, num_flows=4,
    feature_root=32, num_slots=2000, shrink_thres=0.0005,
    mem_usage=[False, True, True, True],
    skip_ops=["none", "concat", "concat"],
)

# Paths
for d in [DATA_DIR, CKPT_DIR, EVAL_DIR]: d.mkdir(parents=True, exist_ok=True)
CKPT_MEMAE  = CKPT_DIR / "ped2_ML_MemAE_SC"
CKPT_HFVAD  = CKPT_DIR / "ped2_HFVAD"
EVAL_MEMAE  = EVAL_DIR / "ped2_ML_MemAE_SC"
EVAL_HFVAD  = EVAL_DIR / "ped2_HFVAD"
for d in [CKPT_MEMAE, CKPT_HFVAD, EVAL_MEMAE, EVAL_HFVAD]: d.mkdir(parents=True, exist_ok=True)

print(f"Frames  : {FRAMES_BASE}")
print(f"Flows   : {FLOWS_BASE}")
print(f"Working : {WORK}")

## STAGE 2 — Ánh xạ dữ liệu Kaggle → cấu trúc HF2VAD

HF2VAD yêu cầu:
```
data/ped2/
  training/frames/Train001/*.tif
  training/flows/Train001/*.tif.npy    ← RAFT .npy đã có
  testing/frames/Test001/*.tif
  testing/flows/Test001/*.tif.npy
  ground_truth_demo/gt_label.json
```
Input Kaggle có cấu trúc:
```
UCSDped2/Train/Train001/*.tif
UCSDped2/Test/Test001/*.tif
optical_flow_results/UCSDped2/Train/Train001/*.tif.npy
optical_flow_results/UCSDped2/Test/Test001/*.tif.npy
```

In [ ]:
def safe_symlink(src: Path, dst: Path):
    """Tạo symlink; bỏ qua nếu đã tồn tại."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and not dst.is_symlink():
        os.symlink(src.resolve(), dst)

def setup_data_structure():
    """Tạo symlink từ Kaggle input → cấu trúc HF2VAD."""
    stats = {}

    for split, kw, hf_sub in [("Train", "Train", "training"), ("Test", "Test", "testing")]:
        frame_src_root = FRAMES_BASE / split
        flow_src_root  = FLOWS_BASE / split
        frame_dst_root = DATA_DIR / hf_sub / "frames"
        flow_dst_root  = DATA_DIR / hf_sub / "flows"

        # ĐÃ SỬA LỖI: Lọc bỏ các thư mục Ground Truth (có đuôi _gt)
        video_dirs = sorted([d for d in frame_src_root.iterdir() if d.is_dir() and not d.name.endswith("_gt")])
        stats[split] = {"videos": len(video_dirs), "frames": 0, "flows": 0}

        for vd in video_dirs:
            # ---- frames ----
            fdst = frame_dst_root / vd.name
            fdst.mkdir(parents=True, exist_ok=True)
            frames = sorted(vd.glob("*.tif"))
            for f in frames:
                safe_symlink(f, fdst / f.name)
            stats[split]["frames"] += len(frames)

            # ---- flows ----
            flow_vd = flow_src_root / vd.name
            if flow_vd.exists():
                fldst = flow_dst_root / vd.name
                fldst.mkdir(parents=True, exist_ok=True)
                flows = sorted(flow_vd.glob("*.npy"))
                for fl in flows:
                    safe_symlink(fl, fldst / fl.name)
                stats[split]["flows"] += len(flows)

    return stats

stats = setup_data_structure()
for split, info in stats.items():
    print(f"{split}: {info['videos']} videos | {info['frames']} frames | {info['flows']} flows")

# Kiểm tra 1 flow sample để biết shape và naming
sample_flows = list((DATA_DIR / "training" / "flows").rglob("*.npy"))
if sample_flows:
    s = np.load(str(sample_flows[0]))
    print(f"\nSample flow: {sample_flows[0].name} → shape={s.shape} dtype={s.dtype}")

In [ ]:
# ---- Ground truth ----
# Ped2 gt: 12 test videos, mỗi video là binary label per frame
# gt_label.json là pickle dict: {0: [0,0,1,...], 1: [...], ...}
# Keys là int 0-11, sorted theo thứ tự video Test001..Test012

GT_PATH = DATA_DIR / "ground_truth_demo" / "gt_label.json"
GT_PATH.parent.mkdir(parents=True, exist_ok=True)

if not GT_PATH.exists():
    print("Tìm kiếm gt masks trong dataset gốc...")
    # Ped2 cung cấp gt masks dạng ảnh binary trong Test*_gt/
    test_frame_root = DATA_DIR / "testing" / "frames"
    test_videos = sorted([d for d in test_frame_root.iterdir() if d.is_dir()])

    gt_dict = {}
    for vid_idx, vd in enumerate(test_videos):
        # Tìm gt folder tương ứng trong input gốc
        vname = vd.name               # e.g. Test001
        gt_src = FRAMES_BASE / "Test" / (vname + "_gt")
        frames_in_vid = sorted(vd.glob("*.tif"))

        if gt_src.exists():
            gt_masks = sorted(gt_src.glob("*.bmp"))
            labels = []
            for m in gt_masks:
                img = cv2.imread(str(m), cv2.IMREAD_GRAYSCALE)
                labels.append(int(img is not None and img.max() > 0))
            # Pad/trim to match frame count
            n = len(frames_in_vid)
            while len(labels) < n: labels.append(0)
            labels = labels[:n]
            gt_dict[vid_idx] = labels
            print(f"  {vname}: {n} frames, {sum(labels)} anomaly")
        else:
            # Nếu không có gt masks → dùng metadata chuẩn Ped2
            # (ground truth public: test 1-12 anomaly intervals)
            gt_dict[vid_idx] = [0] * len(frames_in_vid)
            print(f"  {vname}: gt không tìm thấy, dùng zeros")

    with open(GT_PATH, "wb") as f:
        pickle.dump(gt_dict, f)
    print(f"\n✅ gt_label.json saved: {GT_PATH}")
else:
    gt = pickle.load(open(GT_PATH, "rb"))
    print(f"gt_label.json exists: {len(gt)} videos")
    for k, v in sorted(gt.items())[:3]:
        print(f"  key={k}: {len(v)} frames, {int(sum(v))} anomaly")

## STAGE 3 — Extract BBoxes

Port từ `pre_process/extract_bboxes.py`.

In [ ]:
# ============================================================
# Dataset helper (port từ datasets/dataset.py — ped_dataset)
# ============================================================
def get_inputs(path):
    """Load frame hoặc flow từ file."""
    p = str(path)
    if p.endswith('.npy'):
        arr = np.load(p)              # [H, W, 2]
        if arr.ndim == 2:
            arr = arr[..., np.newaxis]
        return arr
    img = cv2.imread(p)
    if img is None:
        from PIL import Image
        img = np.array(Image.open(p).convert("RGB"))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    # grayscale → BGR
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img


def get_foreground(img, bboxes, patch_size=32):
    """Crop & resize bbox patches."""
    patches = []
    if img.ndim == 3:  # [C, H, W]
        for b in bboxes:
            x1,y1,x2,y2 = int(np.ceil(b[0])),int(np.ceil(b[1])),int(np.ceil(b[2])),int(np.ceil(b[3]))
            p = img[:, y1:y2, x1:x2]
            p = cv2.resize(np.transpose(p,[1,2,0]),(patch_size,patch_size))
            patches.append(np.transpose(p,[2,0,1]))
    elif img.ndim == 4:  # [T, C, H, W]
        for b in bboxes:
            x1,y1,x2,y2 = int(np.ceil(b[0])),int(np.ceil(b[1])),int(np.ceil(b[2])),int(np.ceil(b[3]))
            cube = []
            for t in range(img.shape[0]):
                p = img[t, :, y1:y2, x1:x2]
                p = cv2.resize(np.transpose(p,[1,2,0]),(patch_size,patch_size))
                cube.append(np.transpose(p,[2,0,1]))
            patches.append(np.array(cube))
    return np.array(patches)


class Ped2DS(Dataset):
    """Port từ ped_dataset trong datasets/dataset.py."""
    def __init__(self, data_dir, mode='train', context_frame_num=0,
                 border_mode='hard', file_format='.tif',
                 all_bboxes=None, patch_size=32, of_dataset=False):
        self.data_dir = Path(data_dir)
        self.mode = mode
        self.ctx = context_frame_num
        self.border_mode = border_mode
        self.fmt = file_format
        self.all_bboxes = all_bboxes
        self.patch_size = patch_size
        self.of_dataset = of_dataset
        self.videos = OrderedDict()
        self.all_frame_addr = []
        self.frame_video_idx = []
        self.tot_frame_num = 0
        self._init()

    def _init(self):
        sub  = 'training' if self.mode == 'train' else 'testing'
        kind = 'flows'    if self.of_dataset      else 'frames'
        
        # BÍ QUYẾT: Luôn lấy mốc số lượng từ thư mục frames để làm chuẩn đồng bộ
        frame_root = self.data_dir / sub / 'frames'
        kw   = 'Train'    if self.mode == 'train'  else 'Test'
        video_dirs = sorted([d for d in frame_root.iterdir() if d.is_dir() and kw in d.name])
        
        for idx, vd in enumerate(video_dirs, 1):
            # Lấy danh sách ảnh RGB gốc
            frames = sorted(vd.glob('*' + self.fmt if not self.of_dataset else '*.tif'))
            
            if self.of_dataset:
                # Đang load dataset Flow (.npy)
                flow_dir = self.data_dir / sub / 'flows' / vd.name
                flows = sorted(flow_dir.glob('*.npy'))
                
                addr_list = [str(f) for f in flows]
                
                # Căn chỉnh: Nếu flow ít hơn frame (N-1 flows cho N frames)
                # Nhân bản file flow cuối cùng lên để bù vào khoảng trống
                while len(addr_list) > 0 and len(addr_list) < len(frames):
                    addr_list.append(addr_list[-1])
                
                self.videos[vd.name] = {'frame': addr_list}
                self.frame_video_idx += [idx] * len(frames)
                self.all_frame_addr  += addr_list
            else:
                # Đang load dataset Frame (.tif)
                self.videos[vd.name] = {'frame': [str(f) for f in frames]}
                self.frame_video_idx += [idx] * len(frames)
                self.all_frame_addr  += [str(f) for f in frames]
                
        self.tot_frame_num = len(self.all_frame_addr)

    def __len__(self): return self.tot_frame_num

    def _ctx_range(self, idx):
        """Giống _context_range() trong common_dataset."""
        if self.border_mode == 'predict':
            s = max(0, idx - self.ctx); e = idx
            need = self.ctx + 1
        else:
            s = max(0, idx - self.ctx)
            e = min(self.tot_frame_num - 1, idx + self.ctx)
            need = 2 * self.ctx + 1

        cv   = self.frame_video_idx[idx]
        clip = self.frame_video_idx[s:e+1]
        pad  = need - len(clip)
        if pad > 0:
            clip = ([clip[0]]*pad + clip) if s == 0 else (clip + [clip[-1]]*pad)
        tmp    = np.array(clip) - cv
        offset = int(tmp.sum())

        if pad == 0 and offset == 0:
            return list(range(s, e+1))
        if self.border_mode == 'predict':
            r = list(range(s - offset, e + 1))
            return [r[0]] * max(abs(offset), pad) + r
        if offset > 0:
            r = list(range(s, e - offset + 1)); return r + [r[-1]]*abs(offset)
        elif offset < 0:
            r = list(range(s - offset, e + 1)); return [r[0]]*abs(offset) + r
        if pad > 0:
            r = list(range(s, e+1))
            return ([r[0]]*pad + r) if s == 0 else (r + [r[-1]]*pad)
        return list(range(s, e+1))

    def __getitem__(self, idx):
        fr = self._ctx_range(idx)
        batch = []
        for i in fr:
            img = get_inputs(self.all_frame_addr[i])
            # flows: [H,W,2] → pad to 3ch
            if img.ndim == 3 and img.shape[2] == 2:
                img = np.concatenate([img, np.zeros((*img.shape[:2],1), dtype=img.dtype)], axis=2)
            batch.append(np.transpose(img, [2,0,1]))
        batch = np.array(batch)   # [T, C, H, W]
        if self.all_bboxes is not None:
            batch = get_foreground(batch, self.all_bboxes[idx], self.patch_size)
        return torch.from_numpy(batch.astype(np.float32)), torch.zeros(1)


print("✅ Ped2DS ready")
# Quick sanity check
ds_test = Ped2DS(DATA_DIR, mode='train', context_frame_num=0)
print(f"Train frames: {len(ds_test)}")
ds_test2 = Ped2DS(DATA_DIR, mode='test', context_frame_num=0)
print(f"Test frames : {len(ds_test2)}")

In [ ]:
# ============================================================
# BBox extraction (port từ extract_bboxes.py)
# ============================================================
BBOX_CFG = dict(conf_thr=0.5, min_area=100, cover_thr=0.6,
                binary_thr=18, gauss_k=3, contour_min_area=100)

def get_obj_bboxes(img_bgr, detector, device):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    t   = TF.to_tensor(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        p = detector(t)[0]
    boxes  = p["boxes"].cpu().numpy()
    scores = p["scores"].cpu().numpy()
    labels = p["labels"].cpu().numpy()
    mask = (labels == 1) & (scores >= BBOX_CFG["conf_thr"])
    boxes = boxes[mask]
    if len(boxes) == 0: return np.zeros((0,4), np.float32)
    areas = (boxes[:,2]-boxes[:,0]+1)*(boxes[:,3]-boxes[:,1]+1)
    boxes = boxes[areas >= BBOX_CFG["min_area"]]
    return boxes.astype(np.float32) if len(boxes) else np.zeros((0,4), np.float32)


def del_cover_bboxes(bboxes):
    if len(bboxes) == 0: return bboxes
    x1,y1,x2,y2 = bboxes[:,0],bboxes[:,1],bboxes[:,2],bboxes[:,3]
    areas = (x2-x1+1)*(y2-y1+1)
    order = areas.argsort()
    keep  = []
    for i in range(len(order)):
        si = order[i]; larger = order[i+1:]
        if len(larger) == 0: keep.append(si); continue
        inter = (np.maximum(0, np.minimum(x2[si],x2[larger])-np.maximum(x1[si],x1[larger])+1) *
                 np.maximum(0, np.minimum(y2[si],y2[larger])-np.maximum(y1[si],y1[larger])+1))
        if not np.any(inter/areas[si] > BBOX_CFG["cover_thr"]): keep.append(si)
    return bboxes[keep].astype(np.float32) if keep else np.zeros((0,4), np.float32)


def get_fg_bboxes(imgs_bgr, obj_bboxes):
    gk = BBOX_CFG["gauss_k"]; ext = 2
    sg = np.zeros_like(imgs_bgr[0], np.float32)
    for i in range(len(imgs_bgr)-1):
        g1 = cv2.GaussianBlur(imgs_bgr[i].astype(np.float32),(gk,gk),0)
        g2 = cv2.GaussianBlur(imgs_bgr[i+1].astype(np.float32),(gk,gk),0)
        sg += cv2.absdiff(g1, g2)
    _, bin_ = cv2.threshold(sg.astype(np.uint8), BBOX_CFG["binary_thr"], 255, cv2.THRESH_BINARY)
    for b in obj_bboxes:
        bi = b.astype(int)
        bin_[max(0,bi[1]-ext):min(bi[3]+ext+1,bin_.shape[0]),
             max(0,bi[0]-ext):min(bi[2]+ext+1,bin_.shape[1])] = 0
    gray = cv2.cvtColor(bin_, cv2.COLOR_BGR2GRAY) if bin_.ndim==3 else bin_
    cnts,_ = cv2.findContours(gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    fg = []
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c)
        if (w+1)*(h+1)>BBOX_CFG["contour_min_area"] and w>0 and h>0 and w/h<10 and h/w<10:
            fg.append([max(0,x-ext),max(0,y-ext),
                       min(x+w+ext,gray.shape[1]-1),min(y+h+ext,gray.shape[0]-1)])
    return np.array(fg, np.float32) if fg else np.zeros((0,4), np.float32)


def extract_all_bboxes(mode, detector, device):
    save_p = DATA_DIR / f"ped2_bboxes_{mode}.npy"
    if save_p.exists():
        print(f"  {save_p.name} đã tồn tại, load lại.")
        return np.load(str(save_p), allow_pickle=True)

    ds = Ped2DS(DATA_DIR, mode=mode, context_frame_num=1, border_mode='hard')
    all_bb = []
    for idx in tqdm(range(len(ds)), desc=f"BBoxes [{mode}]"):
        fr = ds._ctx_range(idx)
        imgs = [get_inputs(ds.all_frame_addr[i]) for i in fr]
        imgs = [cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else im for im in imgs]
        cur  = imgs[len(imgs)//2]
        obj  = del_cover_bboxes(get_obj_bboxes(cur, detector, device))
        fg   = get_fg_bboxes(imgs, obj)
        if len(obj) and len(fg): bb = np.concatenate([obj,fg])
        elif len(fg):            bb = fg
        else:                    bb = obj
        all_bb.append(bb)

    all_bb = np.array(all_bb, dtype=object)
    np.save(str(save_p), all_bb)
    print(f"  Saved → {save_p}")
    return all_bb


print("Loading Faster R-CNN...")
det_w    = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=det_w).to(DEVICE).eval()

train_bboxes = extract_all_bboxes('train', detector, DEVICE)
test_bboxes  = extract_all_bboxes('test',  detector, DEVICE)

del detector; gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Train bboxes: {len(train_bboxes)} frames, "
      f"{sum(1 for b in train_bboxes if len(b)>0)} non-empty")
print(f"   Test  bboxes: {len(test_bboxes)} frames, "
      f"{sum(1 for b in test_bboxes  if len(b)>0)} non-empty")

## STAGE 4 — Build Chunked Samples (.pkl)

Port từ `pre_process/extract_samples.py`. Ghép appearance + motion (flow) + bbox → STC, lưu .pkl.

In [ ]:
# Chunked_sample_dataset (port từ datasets/dataset.py)
transform = tvT.Compose([tvT.ToTensor()])

class ChunkedDS(Dataset):
    """Load 1 chunked .pkl file cho training / evaluation."""
    def __init__(self, chunk_file, last_flow=False):
        self.data = joblib.load(chunk_file)
        self.last_flow = last_flow

    def __len__(self): return len(self.data["sample_id"])

    def __getitem__(self, i):
        app       = self.data["appearance"][i]   # [T, C, 32, 32]
        mot       = self.data["motion"][i]        # [T, 2, 32, 32]
        bbox      = self.data["bbox"][i]
        pred_frame= self.data["pred_frame"][i]

        # appearance: [T,C,H,W] → [H,W,T*C] → ToTensor → [T*C,H,W]
        x = np.transpose(app, [2,3,0,1]).reshape(app.shape[2], app.shape[3], -1)  # [H,W,T*C]
        x = transform(x)   # [T*C, H, W]

        # motion: dùng [1:] flows (context_frame_num=4 → 5 frames → 4 flows)
        y = mot[1:] if not self.last_flow else mot[-1:]
        y = np.transpose(y, [2,3,0,1]).reshape(y.shape[2], y.shape[3], -1)  # [H,W,T*2]
        y = transform(y)   # [T*2, H, W]

        return x, y, bbox.astype(np.float32), pred_frame, i


def build_chunked_samples(mode, all_bboxes, chunk_size=100_000):
    """Port từ samples_extraction() trong extract_samples.py."""
    sub     = 'training' if mode == 'train' else 'testing'
    save_dir = DATA_DIR / sub / 'chunked_samples'
    save_dir.mkdir(parents=True, exist_ok=True)

    existing = sorted(save_dir.glob('chunked_samples_*.pkl'))
    if existing:
        print(f"  [{mode}] {len(existing)} chunk file(s) đã có, skip.")
        return save_dir

    # context_frame_num=4, border_mode='predict' → clip 5 frames (4 hist + 1 pred)
    app_ds = Ped2DS(DATA_DIR, mode=mode, context_frame_num=4,
                    border_mode='predict', file_format='.tif',
                    all_bboxes=all_bboxes, patch_size=32, of_dataset=False)
    mot_ds = Ped2DS(DATA_DIR, mode=mode, context_frame_num=4,
                    border_mode='predict', file_format='.npy',
                    all_bboxes=all_bboxes, patch_size=32, of_dataset=True)

    print(f"  [{mode}] Building from {len(app_ds)} frames...")

    gid=0; cnt=0; cid=0
    buf = dict(sample_id=[],appearance=[],motion=[],bbox=[],pred_frame=[])

    def flush():
        nonlocal cid, cnt, buf
        for k in buf: buf[k] = np.array(buf[k])
        out = save_dir / f"chunked_samples_{cid:02d}.pkl"
        joblib.dump(buf, str(out))
        print(f"    Chunk {cid}: {len(buf['sample_id'])} samples → {out.name}")
        cid+=1; cnt=0
        buf = dict(sample_id=[],appearance=[],motion=[],bbox=[],pred_frame=[])

    for idx in tqdm(range(len(app_ds)), desc=f"Chunked [{mode}]"):
        cur_bb = all_bboxes[idx]
        if len(cur_bb) == 0: continue

        fr = app_ds._ctx_range(idx)
        app_t, _ = app_ds[idx]  # [N_bb, T, C, 32, 32]
        mot_t, _ = mot_ds[idx]

        app_np = app_t.numpy()  # [N_bb, T, C, 32, 32]
        mot_np = mot_t.numpy()

        # flows: đảm bảo 2 channels
        if mot_np.ndim == 5:
            if mot_np.shape[2] >= 2: mot_np = mot_np[:, :, :2, :, :]
            elif mot_np.shape[2] == 1: mot_np = np.concatenate([mot_np]*2, axis=2)

        for bi in range(cur_bb.shape[0]):
            buf["sample_id"].append(gid)
            buf["appearance"].append(app_np[bi])
            buf["motion"].append(mot_np[bi])
            buf["bbox"].append(cur_bb[bi])
            buf["pred_frame"].append(fr[-1:])
            gid+=1; cnt+=1
            if cnt == chunk_size: flush()

    if buf["sample_id"]: flush()
    print(f"  ✅ [{mode}] {gid} total samples in {save_dir.name}/")
    return save_dir


train_chunk_dir = build_chunked_samples('train', train_bboxes)
test_chunk_dir  = build_chunked_samples('test',  test_bboxes)

train_chunks = sorted(train_chunk_dir.glob('chunked_samples_*.pkl'))
test_chunks  = sorted(test_chunk_dir.glob('chunked_samples_*.pkl'))
print(f"\nTrain chunks: {[f.name for f in train_chunks]}")
print(f"Test  chunks: {[f.name for f in test_chunks]}")

## STAGE 5 — Model Definitions

Port nguyên xi từ `models/basic_modules.py`, `models/ml_memAE_sc.py`, `models/vunet.py`, `models/mem_cvae.py`.

In [ ]:
# ============================================================
# basic_modules.py
# ============================================================
class SpaceToDepth(nn.Module):
    def __init__(self, bs): super().__init__(); self.bs=bs
    def forward(self,x):
        n,c,h,w=x.size()
        x=x.view(n,c,h//self.bs,self.bs,w//self.bs,self.bs)
        x=x.permute(0,3,5,1,2,4).contiguous()
        return x.view(n,c*(self.bs**2),h//self.bs,w//self.bs)

class DepthToSpace(nn.Module):
    def __init__(self, bs): super().__init__(); self.bs=bs
    def forward(self,x):
        n,c,h,w=x.size()
        x=x.view(n,self.bs,self.bs,c//(self.bs**2),h,w)
        x=x.permute(0,3,4,1,5,2).contiguous()
        return x.view(n,c//(self.bs**2),h*self.bs,w*self.bs)

class IDAct(nn.Module):
    def forward(self,x): return x

class NormConv2d(nn.Module):
    def __init__(self,in_c,out_c,kernel_size,stride=1,padding=0):
        super().__init__()
        self.beta  = nn.Parameter(torch.zeros([1,out_c,1,1]))
        self.gamma = nn.Parameter(torch.ones([1,out_c,1,1]))
        self.conv  = weight_norm(nn.Conv2d(in_c,out_c,kernel_size,stride,padding))
    def forward(self,x): return self.gamma*self.conv(x)+self.beta

class Downsample(nn.Module):
    def __init__(self,ch,out_ch=None,conv_layer=NormConv2d):
        super().__init__()
        out = out_ch or ch
        self.down = conv_layer(ch,out,3,stride=2,padding=1)
    def forward(self,x): return self.down(x)

class Upsample(nn.Module):
    def __init__(self,in_c,out_c,subpixel=True,conv_layer=NormConv2d):
        super().__init__()
        if subpixel:
            self.up  = conv_layer(in_c,4*out_c,3,padding=1)
            self.op2 = DepthToSpace(2)
        else:
            self.up  = nn.ConvTranspose2d(in_c,out_c,3,stride=2,padding=1)
            self.op2 = IDAct()
    def forward(self,x): return self.op2(self.up(x))

class VUnetResnetBlock(nn.Module):
    def __init__(self,out_c,use_skip=False,kernel_size=3,conv_layer=NormConv2d,
                 gated=False,final_act=False,dropout_prob=0.0):
        super().__init__()
        self.dout     = nn.Dropout(p=dropout_prob)
        self.use_skip = use_skip
        self.gated    = gated
        p = kernel_size//2
        if use_skip:
            self.conv2d = conv_layer(2*out_c,out_c,kernel_size,padding=p)
            self.pre    = conv_layer(out_c,out_c,1)
        else:
            self.conv2d = conv_layer(out_c,out_c,kernel_size,padding=p)
        if gated:
            self.conv2d2 = conv_layer(out_c,out_c,kernel_size,padding=p)
            self.dout2   = nn.Dropout(p=dropout_prob)
            self.sigm    = nn.Sigmoid()
        self.act_fn = nn.LeakyReLU() if final_act else nn.ELU()

    def forward(self,x,a=None):
        xp = x
        if self.use_skip:
            a = self.act_fn(self.pre(self.act_fn(a)))
            xp = torch.cat([xp,a],dim=1)
        xp = self.dout(self.act_fn(xp))
        xp = self.conv2d(xp)
        if self.gated:
            xp = self.dout2(self.act_fn(xp))
            xp = self.conv2d2(xp)
            a2,b2 = torch.split(xp,xp.shape[1]//2,1)
            xp = a2*self.sigm(b2)
        return x+xp

print("✅ basic_modules defined")

In [ ]:
# ============================================================
# ML_MemAE_SC (port từ models/ml_memAE_sc.py)
# ============================================================
def hard_shrink_relu(inp, lam=0., eps=1e-12):
    return (F.relu(inp-lam)*inp)/(torch.abs(inp-lam)+eps)

class MemModule(nn.Module):
    def __init__(self,num_slots,slot_dim,shrink_thres=0.0025):
        super().__init__()
        self.shrink_thres = shrink_thres
        self.M = nn.Parameter(torch.empty(num_slots,slot_dim))
        nn.init.uniform_(self.M,-1./math.sqrt(slot_dim),1./math.sqrt(slot_dim))
    def forward(self,x):
        att = F.softmax(F.linear(x,self.M),dim=1)
        if self.shrink_thres>0:
            att = F.normalize(hard_shrink_relu(att,lam=self.shrink_thres),p=1,dim=1)
        return dict(out=F.linear(att,self.M.T),att_weight=att)

# UNet blocks for MemAE
class _DoubleConv(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.c=nn.Sequential(
            nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
            nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.c(x)

class _Down(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.m=nn.Sequential(nn.Conv2d(i,o,3,stride=2,padding=1),_DoubleConv(o,o))
    def forward(self,x): return self.m(x)

class _Up(nn.Module):
    def __init__(self,i,o,op='none'):
        super().__init__()
        self.op=op
        self.up=nn.ConvTranspose2d(i,i//2,3,stride=2,padding=1,output_padding=1)
        self.cv=_DoubleConv(i if op=='concat' else i//2, o)
    def forward(self,x1,x2=None):
        x1=self.up(x1)
        if self.op=='concat' and x2 is not None: x1=torch.cat([x2,x1],1)
        return self.cv(x1)

class ML_MemAE_SC(nn.Module):
    def __init__(self,num_in_ch,seq_len,features_root,
                 num_slots,shrink_thres,mem_usage,skip_ops):
        super().__init__()
        fr=features_root
        self.enc0 = _DoubleConv(num_in_ch*seq_len, fr)
        self.dn1  = _Down(fr,   fr*2)
        self.dn2  = _Down(fr*2, fr*4)
        self.dn3  = _Down(fr*4, fr*8)

        self.mem1 = MemModule(num_slots,fr*2 *16*16,shrink_thres) if mem_usage[1] else None
        self.mem2 = MemModule(num_slots,fr*4 * 8* 8,shrink_thres) if mem_usage[2] else None
        self.mem3 = MemModule(num_slots,fr*8 * 4* 4,shrink_thres) if mem_usage[3] else None

        self.up3  = _Up(fr*8, fr*4, op=skip_ops[-1])
        self.up2  = _Up(fr*4, fr*2, op=skip_ops[-2])
        self.up1  = _Up(fr*2, fr,   op=skip_ops[-3])
        self.out  = nn.Conv2d(fr, num_in_ch*seq_len, 1)
        self.mem_usage=mem_usage; self.skip_ops=skip_ops

    def _mem(self, mem, feat):
        if mem is None: return feat, torch.zeros(feat.size(0),1,device=feat.device)
        b,c,h,w = feat.shape
        r = mem(feat.view(b,-1))
        return r['out'].view(b,c,h,w), r['att_weight']

    def forward(self,x):
        x1=self.enc0(x); x2=self.dn1(x1); x3=self.dn2(x2); x4=self.dn3(x3)
        x2,a1=self._mem(self.mem1,x2)
        x3,a2=self._mem(self.mem2,x3)
        x4,a3=self._mem(self.mem3,x4)
        x=self.up3(x4, x3 if self.skip_ops[-1]=='concat' else None)
        x=self.up2(x,  x2 if self.skip_ops[-2]=='concat' else None)
        x=self.up1(x,  x1 if self.skip_ops[-3]=='concat' else None)
        return dict(recon=self.out(x),att_weight3=a3,att_weight2=a2,att_weight1=a1)

print("✅ ML_MemAE_SC defined")

In [ ]:
# ============================================================
# VUNet — port từ models/vunet.py (FULL, nguyên bản)
# ============================================================
class VUnetEncoder(nn.Module):
    def __init__(self,n_stages,nf_in=3,nf_start=64,nf_max=128,n_rnb=2,
                 conv_layer=NormConv2d,dropout_prob=0.0):
        super().__init__()
        self.in_op=conv_layer(nf_in,nf_start,kernel_size=1)
        nf=nf_start; self.blocks=ModuleDict(); self.downs=ModuleDict()
        self.n_rnb=n_rnb; self.n_stages=n_stages
        for i_s in range(n_stages):
            if i_s>0:
                self.downs[f"s{i_s+1}"]=Downsample(nf,min(2*nf,nf_max),conv_layer=conv_layer)
                nf=min(2*nf,nf_max)
            for ir in range(n_rnb):
                self.blocks[f"s{i_s+1}_{ir+1}"]=VUnetResnetBlock(nf,conv_layer=conv_layer,dropout_prob=dropout_prob)
    def forward(self,x):
        out={}; h=self.in_op(x)
        for ir in range(self.n_rnb): h=self.blocks[f"s1_{ir+1}"](h); out[f"s1_{ir+1}"]=h
        for i_s in range(1,self.n_stages):
            h=self.downs[f"s{i_s+1}"](h)
            for ir in range(self.n_rnb):
                h=self.blocks[f"s{i_s+1}_{ir+1}"](h); out[f"s{i_s+1}_{ir+1}"]=h
        return out

class ZConverter(nn.Module):
    def __init__(self,n_stages,nf,device,conv_layer=NormConv2d,dropout_prob=0.0):
        super().__init__()
        self.n_stages=n_stages; self.device=device
        self.blocks=ModuleList([VUnetResnetBlock(nf,use_skip=True,conv_layer=conv_layer,dropout_prob=dropout_prob) for _ in range(3)])
        self.conv1x1=conv_layer(nf,nf,1)
        self.up=Upsample(nf,nf,conv_layer=conv_layer)
        self.channel_norm=conv_layer(2*nf,nf,1)
        self.d2s=DepthToSpace(2); self.s2d=SpaceToDepth(2)
    def _sample(self,mean):
        return mean+torch.randn_like(mean)
    def forward(self,x_f):
        params={}; zs={}
        h=self.conv1x1(x_f[f"s{self.n_stages}_2"])
        for n,i_s in enumerate(range(self.n_stages,self.n_stages-2,-1)):
            stage=f"s{i_s}"; sp=x_f[stage+"_2"].shape[-1]
            skey="%dby%d"%(sp,sp)
            h=self.blocks[2*n](h,x_f[stage+"_2"])
            params[skey]=h; z=self._sample(h); zs[skey]=z
            if n==0:
                gz=self.channel_norm(torch.cat([x_f[stage+"_1"],z],1))
                h=self.blocks[2*n+1](h,gz); h=self.up(h)
        return params,zs

class VUnetBottleneck(nn.Module):
    def __init__(self,n_stages,nf,device,n_rnb=2,n_auto_groups=4,conv_layer=NormConv2d,dropout_prob=0.0):
        super().__init__()
        self.device=device; self.n_stages=n_stages; self.n_rnb=n_rnb; self.n_auto_groups=n_auto_groups
        self.blocks=ModuleDict(); self.channel_norm=ModuleDict()
        self.conv1x1=conv_layer(nf,nf,1)
        self.up=Upsample(nf,nf,conv_layer=conv_layer)
        self.d2s=DepthToSpace(2); self.s2d=SpaceToDepth(2)
        for i_s in range(n_stages,n_stages-2,-1):
            self.channel_norm[f"s{i_s}"]=conv_layer(2*nf,nf,1)
            for ir in range(n_rnb):
                self.blocks[f"s{i_s}_{ir+1}"]=VUnetResnetBlock(nf,use_skip=True,conv_layer=conv_layer,dropout_prob=dropout_prob)
        self.auto_blocks=ModuleList()
        for i_a in range(4):
            if i_a<1:
                self.auto_blocks.append(VUnetResnetBlock(nf,conv_layer=conv_layer,dropout_prob=dropout_prob))
                self.param_converter=conv_layer(4*nf,nf,kernel_size=1)
            else:
                self.auto_blocks.append(VUnetResnetBlock(nf,use_skip=True,conv_layer=conv_layer,dropout_prob=dropout_prob))
    def _sample(self,mean): return mean+torch.randn(mean.size(),device=self.device)
    def _merge(self,x): return self.d2s(torch.cat(x,1))
    def forward(self,x_e,z_post):
        p_params={}; z_prior={}; use_z=True
        h=self.conv1x1(x_e[f"s{self.n_stages}_2"])
        for i_s in range(self.n_stages,self.n_stages-2,-1):
            stage=f"s{i_s}"; sp=x_e[stage+"_2"].shape[-1]; skey="%dby%d"%(sp,sp)
            h=self.blocks[stage+"_2"](h,x_e[stage+"_2"])
            if sp==1:
                p_params[skey]=h; ps=self._sample(h); z_prior[skey]=ps
            else:
                if use_z:
                    zf=self.s2d(z_post[skey]) if z_post[skey].shape[2]>1 else z_post[skey]
                    ss2=zf.shape[1]//4; zg=torch.split(zf,[ss2,ss2,ss2,ss2],1)
                pg=[]; sg=[]
                pf=self.param_converter(self.s2d(self.auto_blocks[0](h)))
                for i_a in range(len(self.auto_blocks)):
                    pg.append(pf); ps2=self._sample(pg[-1]); sg.append(ps2)
                    if i_a+1<len(self.auto_blocks):
                        fb=zg[i_a] if use_z else ps2
                        pf=self.auto_blocks[i_a+1](pf,fb)
                p_params[skey]=self._merge(pg); z_prior[skey]=self._merge(sg)
            z=(self.d2s(z_post[skey]) if z_post[skey].shape[-1]!=h.shape[-1] else z_post[skey]) if use_z else z_prior[skey]
            gz=self.channel_norm[stage](torch.cat([x_e[stage+"_1"],z],1))
            h=self.blocks[stage+"_1"](h,gz)
            if i_s==self.n_stages: h=self.up(h)
        return h,p_params,z_prior

class VUnetDecoder(nn.Module):
    def __init__(self,n_stages,nf=128,nf_out=3,n_rnb=2,conv_layer=NormConv2d,
                 spatial_size=256,final_act=True,dropout_prob=0.0):
        super().__init__()
        self.blocks=ModuleDict(); self.ups=ModuleDict()
        self.n_stages=n_stages; self.n_rnb=n_rnb
        for i_s in range(n_stages-2,0,-1):
            out_nf=nf//2 if i_s==1 else nf
            self.ups[f"s{i_s+1}"]=Upsample(nf,out_nf,conv_layer=conv_layer)
            nf=out_nf
            for ir in range(n_rnb,0,-1):
                self.blocks[f"s{i_s}_{ir}"]=VUnetResnetBlock(nf,use_skip=True,conv_layer=conv_layer,dropout_prob=dropout_prob)
        self.final_layer=conv_layer(nf,nf_out,kernel_size=1)
        self.final_act=nn.Sigmoid()
    def forward(self,x,skips):
        out=x
        for i_s in range(self.n_stages-2,0,-1):
            out=self.ups[f"s{i_s+1}"](out)
            for ir in range(self.n_rnb,0,-1): out=self.blocks[f"s{i_s}_{ir}"](out,skips[f"s{i_s}_{ir}"])
        return self.final_act(self.final_layer(out))

class VUnet(nn.Module):
    def __init__(self,config):
        super().__init__()
        mp=config['model_paras']; dev=config.get('device','cuda')
        fa=mp.get('final_act',False); nfm=mp['nf_max']; nfs=mp['nf_start']
        sp=mp['spatial_size']; dp=mp['dropout_prob']
        ic=mp['img_channels']; mc=mp['motion_channels']
        ch=mp['clip_hist']; cp=mp['clip_pred']; nf=mp['num_flows']
        n_s=1+int(np.round(np.log2(sp)))-2
        cl=Conv2d if fa else NormConv2d
        self.f_phi=VUnetEncoder(n_s,ic*ch+mc*nf,nfs,nfm,conv_layer=cl,dropout_prob=dp)
        self.e_theta=VUnetEncoder(n_s,mc*nf,nfs,nfm,conv_layer=cl,dropout_prob=dp)
        self.zc=ZConverter(n_s,nfm,dev,conv_layer=cl,dropout_prob=dp)
        self.bottleneck=VUnetBottleneck(n_s,nfm,dev,conv_layer=cl,dropout_prob=dp)
        self.decoder=VUnetDecoder(n_s,nfm,ic*cp,conv_layer=cl,spatial_size=sp,final_act=fa,dropout_prob=dp)
        self.saved_tensors=None
    def forward(self,inputs,mode='train'):
        xf_in=torch.cat((inputs['appearance'],inputs['motion']),1)
        x_f=self.f_phi(xf_in)
        q_means,zs=self.zc(x_f)
        x_e=self.e_theta(inputs['motion'])
        out_b,p_means,ps=self.bottleneck(x_e,zs if mode=='train' else q_means)
        out_img=self.decoder(out_b,x_f)
        self.saved_tensors=dict(q_means=q_means,p_means=p_means)
        return out_img

print("✅ VUNet defined")

In [ ]:
# ============================================================
# HFVAD = ML_MemAE_SC + VUNet (port từ models/mem_cvae.py)
# ============================================================
class HFVAD(nn.Module):
    def __init__(self,num_hist,num_pred,config,features_root,
                 num_slots,shrink_thres,skip_ops,mem_usage):
        super().__init__()
        self.num_hist=num_hist; self.num_pred=num_pred
        self.memAE=ML_MemAE_SC(num_in_ch=2,seq_len=1,
            features_root=features_root,num_slots=num_slots,
            shrink_thres=shrink_thres,mem_usage=mem_usage,skip_ops=skip_ops)
        self.vunet=VUnet(config)

    def forward(self,frames,flows,mode='train'):
        of_recon=torch.zeros_like(flows)
        a3c,a2c,a1c=[],[],[]
        for j in range(self.num_hist):
            out=self.memAE(flows[:,2*j:2*(j+1)])
            of_recon[:,2*j:2*(j+1)]=out['recon']
            a3c.append(out['att_weight3']); a2c.append(out['att_weight2']); a1c.append(out['att_weight1'])
        frame_in  =frames[:,:-3*self.num_pred]
        frame_tgt =frames[:,-3*self.num_pred:]
        pred=self.vunet(dict(appearance=frame_in,motion=of_recon),mode=mode)
        out=dict(frame_pred=pred,frame_target=frame_tgt,
                 of_recon=of_recon,of_target=flows)
        out.update(self.vunet.saved_tensors)
        return out


# ---- Loss functions (port từ losses/loss.py) ----
def kl_loss(q_means, p_means):
    loss=0.
    for k in q_means:
        kl=0.5*torch.pow(q_means[k]-p_means[k],2)
        loss+=torch.mean(torch.sum(kl,dim=[1,2,3]))
    return loss

class IntensityLoss(nn.Module):
    def __init__(self,l_num): super().__init__(); self.l=l_num
    def forward(self,p,t): return torch.mean(torch.abs((p-t)**self.l))

class GradLoss(nn.Module):
    def __init__(self,alpha,ch,device):
        super().__init__(); self.alpha=alpha
        f=torch.FloatTensor([[-1.,1.]]).to(device)
        self.fx=f.view(1,1,1,2).repeat(1,ch,1,1)
        self.fy=f.view(1,1,2,1).repeat(1,ch,1,1)
    def forward(self,p,t):
        gx=F.conv2d(F.pad(p,(1,0,0,0)),self.fx)-F.conv2d(F.pad(t,(1,0,0,0)),self.fx)
        gy=F.conv2d(F.pad(p,(0,0,1,0)),self.fy)-F.conv2d(F.pad(t,(0,0,1,0)),self.fy)
        return torch.mean(torch.abs(gx)**self.alpha+torch.abs(gy)**self.alpha)

# ---- Checkpoint helpers ----
def compat_load(p): return torch.load(p, weights_only=False, map_location='cpu')
def save_ckpt(state_dict, opt_state, path, epoch, step):
    torch.save({'model_state_dict':state_dict,'optimizer_state_dict':opt_state,'step':step}, f"{path}-{epoch}")
def save_model(state_dict, path):
    torch.save({'model_state_dict':state_dict}, path)

def weights_init_kaiming(m):
    if isinstance(m,(nn.Conv2d,nn.Linear)):
        nn.init.kaiming_normal_(m.weight.data,a=0.1,mode='fan_in')
    elif isinstance(m,nn.BatchNorm2d):
        nn.init.normal_(m.weight.data,1.0,0.02)
        nn.init.constant_(m.bias.data,0.0)

print("✅ HFVAD + losses defined")

# Quick architecture check
cfg_tmp = {'model_paras': MODEL_PARAS, 'device': DEVICE}
_m = ML_MemAE_SC(2,1,MODEL_PARAS['feature_root'],MODEL_PARAS['num_slots'],
                 MODEL_PARAS['shrink_thres'],MODEL_PARAS['mem_usage'],MODEL_PARAS['skip_ops'])
_x = torch.randn(2,2,32,32)
_o = _m(_x); print(f"MemAE output shape: {_o['recon'].shape}")
del _m,_x,_o; gc.collect()

## STAGE 6 — Train Stage-1: ML-MemAE-SC

Port từ `ml_memAE_sc_train.py`. Train flow reconstruction autoencoder với memory.

In [ ]:
# ============================================================
# AUC evaluation helper (port từ ml_memAE_sc_eval.py + eval.py)
# ============================================================
def compute_auc(frame_bbox_scores, gt_path, dataset_name='ped2', eval_save_dir=None, suffix=''):
    """Tính AUC từ per-frame anomaly scores và ground-truth labels."""
    gt = pickle.load(open(gt_path,'rb'))
    # Sort by key (int)
    sorted_gt = sorted(gt.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else x[0])
    testing_frame_counts = [len(np.asarray(v)) for _,v in sorted_gt]
    gt_concat = np.concatenate([np.asarray(v) for _,v in sorted_gt])

    # frame-level score = max over bboxes
    frame_scores = np.zeros(len(frame_bbox_scores))
    for i,d in enumerate(frame_bbox_scores):
        frame_scores[i] = max(d.values()) if d else 0.0

    # Trim first 4 frames per video (border effect)
    new_gt, new_sc = [], []
    start = 0
    for n in testing_frame_counts:
        new_gt.append(gt_concat[start:start+n][4:])
        new_sc.append(frame_scores[start:start+n][4:])
        start += n
    gt_concat   = np.concatenate(new_gt)
    frame_scores= np.concatenate(new_sc)

    # Median filter + ROC
    eff_counts = [max(0, n-4) for n in testing_frame_counts]
    ptr = 0
    smooth_scores = []
    for n in eff_counts:
        seg = frame_scores[ptr:ptr+n]
        if len(seg): seg = ss.medfilt(seg, kernel_size=17)
        smooth_scores.append(seg); ptr += n
    frame_scores_sm = np.concatenate(smooth_scores) if smooth_scores else frame_scores

    fpr, tpr, _ = roc_curve(gt_concat, frame_scores_sm, pos_label=1)
    auroc = sk_auc(fpr, tpr)

    # Save ROC curve
    if eval_save_dir:
        Path(eval_save_dir).mkdir(parents=True, exist_ok=True)
        fig,ax=plt.subplots()
        ax.plot(fpr,tpr,label=f'AUC={auroc:.4f}'); ax.plot([0,1],[0,1],'--')
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend()
        ax.set_title(f'ROC {suffix}'); fig.savefig(str(Path(eval_save_dir)/f'roc_{suffix}.png'))
        plt.close(fig)

    return auroc


def eval_memae(model, test_chunks, gt_path, device, eval_batchsize=32):
    """Evaluate ML-MemAE-SC AUC (port từ ml_memAE_sc_eval.py)."""
    model.eval()
    score_fn = nn.MSELoss(reduction='none')
    gt = pickle.load(open(gt_path,'rb'))
    sorted_gt = sorted(gt.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else x[0])
    total_frames = sum(len(np.asarray(v)) for _,v in sorted_gt)
    frame_bbox_scores = [{} for _ in range(total_frames)]
    uid = 0
    with torch.no_grad():
        for cf in test_chunks:
            ds = ChunkedDS(cf, last_flow=True)
            dl = DataLoader(ds, batch_size=eval_batchsize, shuffle=False)
            for _,ofs,_,pred_f,_ in tqdm(dl, desc='Eval MemAE', leave=False):
                ofs = ofs.to(device)
                out = model(ofs)
                loss = score_fn(out['recon'], ofs).cpu().numpy()
                sc = np.sum(loss, axis=(1,2,3))
                for i,s in enumerate(sc):
                    fid = int(pred_f[i][-1].item())
                    frame_bbox_scores[fid][uid] = s; uid += 1
    return compute_auc(frame_bbox_scores, gt_path, eval_save_dir=str(EVAL_MEMAE))


print("✅ Evaluation helpers ready")

In [ ]:
# ============================================================
# TRAIN STAGE-1: ML-MemAE-SC
# port từ ml_memAE_sc_train.py::train()
# ============================================================
print("=" * 60)
print("STAGE-1: Training ML-MemAE-SC")
print(f"  Epochs: {EPOCHS_MEMAE} | Batch: {BATCH_SIZE} | LR: {LR}")
print("=" * 60)

memae = ML_MemAE_SC(
    num_in_ch=MODEL_PARAS['motion_channels'],
    seq_len=1,
    features_root=MODEL_PARAS['feature_root'],
    num_slots=MODEL_PARAS['num_slots'],
    shrink_thres=MODEL_PARAS['shrink_thres'],
    mem_usage=MODEL_PARAS['mem_usage'],
    skip_ops=MODEL_PARAS['skip_ops'],
).to(DEVICE)
memae.apply(weights_init_kaiming)

mse_loss = nn.MSELoss().to(DEVICE)
opt_m = optim.Adam(memae.parameters(), lr=LR, eps=1e-7)
sch_m = optim.lr_scheduler.MultiStepLR(opt_m, milestones=[50], gamma=0.8)

best_auc_m = -1.0
step_m = 0

for epoch in range(EPOCHS_MEMAE):
    for cf in sorted(train_chunk_dir.glob('chunked_samples_*.pkl')):
        ds  = ChunkedDS(cf, last_flow=True)
        dl  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        for _,ofs,_,_,_ in tqdm(dl, desc=f"MemAE E{epoch+1}"):
            memae.train()
            ofs = ofs.to(DEVICE)
            out = memae(ofs)

            l_recon   = mse_loss(out['recon'], ofs)
            l_sparse  = (
                torch.mean(torch.sum(-out['att_weight3']*torch.log(out['att_weight3']+1e-12),1))
              + torch.mean(torch.sum(-out['att_weight2']*torch.log(out['att_weight2']+1e-12),1))
              + torch.mean(torch.sum(-out['att_weight1']*torch.log(out['att_weight1']+1e-12),1))
            )
            loss = 1.0*l_recon + 0.0002*l_sparse

            opt_m.zero_grad(); loss.backward(); opt_m.step()
            if step_m % 200 == 199:
                print(f"  [E{epoch+1} S{step_m+1}] loss={loss.item():.4f} "
                      f"recon={l_recon.item():.4f} sparse={l_sparse.item():.4f}")
            step_m += 1
        del ds, dl; gc.collect(); torch.cuda.empty_cache()

    sch_m.step()

    # Save checkpoint
    ckpt_path = str(CKPT_MEMAE / "model.pth")
    save_ckpt(memae.state_dict(), opt_m.state_dict(), ckpt_path, epoch+1, step_m)

    # Evaluate AUC
    with torch.no_grad():
        auc_m = eval_memae(memae, sorted(test_chunk_dir.glob('chunked_samples_*.pkl')),
                           str(GT_PATH), DEVICE)
    print(f"  → Epoch {epoch+1} AUC (MemAE): {auc_m:.4f}")
    if auc_m > best_auc_m:
        best_auc_m = auc_m
        save_model(memae.state_dict(), str(CKPT_MEMAE / "best.pth"))
        print(f"  ✅ Best MemAE saved! AUC={best_auc_m:.4f}")

print(f"\n{'='*60}")
print(f"Stage-1 best AUC (ML-MemAE-SC): {best_auc_m:.4f}")
print(f"{'='*60}")

## STAGE 7 — Train Stage-2: HFVAD (ML-MemAE-SC + VUNet)

Port từ `train.py`. Freeze MemAE, chỉ train VUNet.

In [ ]:
# ============================================================
# Eval HFVAD full model (port từ eval.py::evaluate())
# ============================================================
def eval_hfvad(model, test_chunks, gt_path, train_stats_path, device, eval_batchsize=32, suffix=''):
    model.eval()
    score_fn = nn.MSELoss(reduction='none')

    # Training stats for normalisation
    if train_stats_path and Path(train_stats_path).exists():
        stats = compat_load(train_stats_path)
        of_mu, of_std     = stats['of_training_stats'].mean(),  stats['of_training_stats'].std()
        fr_mu, fr_std     = stats['frame_training_stats'].mean(),stats['frame_training_stats'].std()
        do_norm = True
    else:
        do_norm = False

    gt = pickle.load(open(gt_path,'rb'))
    sorted_gt = sorted(gt.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else x[0])
    total_frames = sum(len(np.asarray(v)) for _,v in sorted_gt)
    frame_bbox_scores = [{} for _ in range(total_frames)]
    uid = 0

    with torch.no_grad():
        for cf in test_chunks:
            ds = ChunkedDS(cf, last_flow=False)
            dl = DataLoader(ds, batch_size=eval_batchsize, shuffle=False)
            for frames,ofs,_,pred_f,_ in tqdm(dl, desc=f'Eval HFVAD {suffix}', leave=False):
                frames,ofs = frames.to(device), ofs.to(device)
                out = model(frames, ofs, mode='test')
                l_of = score_fn(out['of_recon'], out['of_target']).cpu().numpy()
                l_fr = score_fn(out['frame_pred'], out['frame_target']).cpu().numpy()
                sc_of = np.sum(l_of,(1,2,3))
                sc_fr = np.sum(l_fr,(1,2,3))
                if do_norm:
                    sc_of = (sc_of-of_mu)/of_std
                    sc_fr = (sc_fr-fr_mu)/fr_std
                sc = 1.0*sc_of + 0.1*sc_fr
                for i,s in enumerate(sc):
                    fid = int(pred_f[i][-1].item())
                    frame_bbox_scores[fid][uid]=s; uid+=1
            del ds,dl; gc.collect(); torch.cuda.empty_cache()

    return compute_auc(frame_bbox_scores, gt_path,
                       eval_save_dir=str(EVAL_HFVAD), suffix=suffix)


def calc_training_stats(model, train_chunks, device, save_path, batch_size=32):
    """Port từ cal_training_stats() trong train.py."""
    model.eval()
    score_fn = nn.MSELoss(reduction='none')
    of_stats, fr_stats = [], []
    with torch.no_grad():
        for cf in train_chunks:
            ds = ChunkedDS(cf, last_flow=False)
            dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
            for frames,ofs,_,_,_ in tqdm(dl, desc='Train stats', leave=False):
                frames,ofs = frames.to(device), ofs.to(device)
                out = model(frames, ofs, mode='test')
                of_stats.append(np.sum(score_fn(out['of_recon'],out['of_target']).cpu().numpy(),(1,2,3)))
                fr_stats.append(np.sum(score_fn(out['frame_pred'],out['frame_target']).cpu().numpy(),(1,2,3)))
            del ds,dl; gc.collect(); torch.cuda.empty_cache()
    torch.save({'of_training_stats':np.concatenate(of_stats),
                'frame_training_stats':np.concatenate(fr_stats)}, save_path)
    print(f"  Training stats saved → {save_path}")


print("✅ HFVAD eval helpers ready")

In [ ]:
# ============================================================
# TRAIN STAGE-2: HFVAD full model
# port từ train.py::train()
# ============================================================
print("=" * 60)
print("STAGE-2: Training HFVAD (MemAE frozen + VUNet)")
print(f"  Epochs: {EPOCHS_HFVAD} | Batch: {BATCH_SIZE} | LR: {LR}")
print("=" * 60)

cfg_for_model = {'model_paras': MODEL_PARAS, 'device': DEVICE}
hfvad = HFVAD(
    num_hist=MODEL_PARAS['clip_hist'],
    num_pred=MODEL_PARAS['clip_pred'],
    config=cfg_for_model,
    features_root=MODEL_PARAS['feature_root'],
    num_slots=MODEL_PARAS['num_slots'],
    shrink_thres=MODEL_PARAS['shrink_thres'],
    mem_usage=MODEL_PARAS['mem_usage'],
    skip_ops=MODEL_PARAS['skip_ops'],
).to(DEVICE)

# Load Stage-1 MemAE weights → freeze
memae_best = CKPT_MEMAE / "best.pth"
if memae_best.exists():
    state = compat_load(str(memae_best))['model_state_dict']
    hfvad.memAE.load_state_dict(state)
    print(f"  ✅ Loaded MemAE weights from {memae_best.name}")
else:
    print("  ⚠️  best.pth not found, using last checkpoint")
    last_ckpt = sorted((CKPT_MEMAE).glob('model.pth-*'))[-1]
    state = compat_load(str(last_ckpt))['model_state_dict']
    hfvad.memAE.load_state_dict(state)

for p in hfvad.memAE.parameters(): p.requires_grad_(False)
hfvad.memAE.eval()
hfvad.vunet.apply(weights_init_kaiming)

grad_loss_fn = GradLoss(alpha=1, ch=3*MODEL_PARAS['clip_pred'], device=DEVICE).to(DEVICE)
intensity_fn = IntensityLoss(l_num=2).to(DEVICE)

opt_h = optim.Adam(hfvad.vunet.parameters(), lr=LR, eps=1e-7)
sch_h = optim.lr_scheduler.StepLR(opt_h, step_size=50, gamma=0.8)

best_auc_h = -1.0
step_h = 0
train_stats_path = str(CKPT_HFVAD / "training_stats.npy")

for epoch in range(EPOCHS_HFVAD):
    for cf in sorted(train_chunk_dir.glob('chunked_samples_*.pkl')):
        ds = ChunkedDS(cf, last_flow=False)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        for frames,ofs,_,_,_ in tqdm(dl, desc=f"HFVAD E{epoch+1}"):
            hfvad.vunet.train()
            frames, ofs = frames.to(DEVICE), ofs.to(DEVICE)
            out = hfvad(frames, ofs, mode='train')

            l_kl   = kl_loss(out['q_means'], out['p_means'])
            l_fr   = intensity_fn(out['frame_pred'], out['frame_target'])
            l_grad = grad_loss_fn(out['frame_pred'], out['frame_target'])
            loss   = 1.0*l_kl + 1.0*l_fr + 1.0*l_grad

            opt_h.zero_grad(); loss.backward(); opt_h.step()
            if step_h % 200 == 199:
                print(f"  [E{epoch+1} S{step_h+1}] loss={loss.item():.4f} "
                      f"kl={l_kl.item():.4f} fr={l_fr.item():.4f} grad={l_grad.item():.4f}")
            step_h += 1
        del ds,dl; gc.collect(); torch.cuda.empty_cache()

    sch_h.step()

    # Save checkpoint
    ckpt_h_path = str(CKPT_HFVAD / "model.pth")
    save_ckpt(hfvad.state_dict(), opt_h.state_dict(), ckpt_h_path, epoch+1, step_h)

    # Training stats (for score normalisation)
    with torch.no_grad():
        calc_training_stats(hfvad, sorted(train_chunk_dir.glob('*.pkl')),
                            DEVICE, train_stats_path)

    # Evaluate AUC
    with torch.no_grad():
        auc_h = eval_hfvad(hfvad,
                           sorted(test_chunk_dir.glob('chunked_samples_*.pkl')),
                           str(GT_PATH), train_stats_path, DEVICE,
                           suffix=str(epoch+1))
    print(f"  → Epoch {epoch+1} AUC (HFVAD): {auc_h:.4f}")

    if auc_h > best_auc_h:
        best_auc_h = auc_h
        save_model(hfvad.state_dict(), str(CKPT_HFVAD / "best.pth"))
        print(f"  ✅ Best HFVAD saved! AUC={best_auc_h:.4f}")

print(f"\n{'='*60}")
print(f"Stage-2 best AUC (HFVAD): {best_auc_h:.4f}")
print(f"{'='*60}")

## STAGE 8 — Final Evaluation & Visualization

In [ ]:
# ============================================================
# Load best model và evaluate final AUC
# ============================================================
print("Loading best HFVAD checkpoint...")
best_ckpt = CKPT_HFVAD / "best.pth"
if best_ckpt.exists():
    cfg_for_model = {'model_paras': MODEL_PARAS, 'device': DEVICE}
    hfvad_best = HFVAD(
        num_hist=MODEL_PARAS['clip_hist'], num_pred=MODEL_PARAS['clip_pred'],
        config=cfg_for_model, features_root=MODEL_PARAS['feature_root'],
        num_slots=MODEL_PARAS['num_slots'], shrink_thres=MODEL_PARAS['shrink_thres'],
        mem_usage=MODEL_PARAS['mem_usage'], skip_ops=MODEL_PARAS['skip_ops'],
    ).to(DEVICE).eval()
    hfvad_best.load_state_dict(compat_load(str(best_ckpt))['model_state_dict'])

    with torch.no_grad():
        final_auc = eval_hfvad(
            hfvad_best,
            sorted(test_chunk_dir.glob('chunked_samples_*.pkl')),
            str(GT_PATH), train_stats_path, DEVICE, suffix='final'
        )
    print(f"\n{'='*60}")
    print(f"  FINAL AUC (HFVAD, best ckpt): {final_auc:.4f}")
    print(f"  (paper reports ~93% with 50 epochs; ~few epochs < 80%)")
    print(f"{'='*60}")
else:
    print("best.pth không tồn tại — dùng model cuối.")
    final_auc = best_auc_h
    print(f"Last epoch AUC: {final_auc:.4f}")

In [ ]:
# ============================================================
# Hiển thị ROC curve
# ============================================================
roc_files = sorted(EVAL_HFVAD.glob('roc_*.png'))
if roc_files:
    from PIL import Image
    latest = roc_files[-1]
    img = np.array(Image.open(str(latest)))
    plt.figure(figsize=(6,5))
    plt.imshow(img); plt.axis('off')
    plt.title(f'ROC Curve — AUC={final_auc:.4f}')
    plt.tight_layout(); plt.show()
    print(f"ROC saved: {latest}")
else:
    print("Chưa có ROC file.")

In [ ]:
# ============================================================
# Summary
# ============================================================
print("\n" + "="*60)
print(" SUMMARY")
print("="*60)
print(f" Dataset     : UCSDped2 (RAFT optical flow)")
print(f" Stage-1 AUC : {best_auc_m:.4f}  (ML-MemAE-SC, {EPOCHS_MEMAE} epochs)")
print(f" Stage-2 AUC : {best_auc_h:.4f}  (HFVAD full, {EPOCHS_HFVAD} epochs)")
print(f" Final  AUC  : {final_auc:.4f}")
print()
print(" Thay đổi so với pipeline gốc:")
print("  • FlowNet2 (CUDA custom ops)    → RAFT-Large (torchvision)")
print("  • Cascade RCNN (mmdet)          → Faster R-CNN (torchvision)")
print("  • Optical flow: đã extract sẵn  → load trực tiếp .npy")
print("  • Output .npy format: [H,W,2]   → giống nhau")
print("="*60)